# Amidst Q1–Q5 W-A6 Colab 一鍵測試 Notebook（環境自檢 → 資料 SHA-256 → 中斷續跑測試 → 結果匯出）

本 Notebook 提供 **Run all 一鍵跑通** 的標準化實驗流程，涵蓋：
1. **Cell 1 — 環境自檢**：收集 OS、CPU、RAM、GPU/VRAM、Python/PyTorch 版本，產出並驗證 `amidst.experiment-env.v1` 環境紀錄。
2. **Cell 2 — API Key 安全檢查**：僅從 Colab Secrets (`google.colab.userdata`) 或環境變數檢查金鑰狀態 (`PRESENT` / `MISSING`)，**絕不印出或記錄任何金鑰值**。
3. **Cell 3 — Google Drive 掛載與 W-C5 safe 資料包 SHA-256 驗證**：驗證 SHA-256 並強制檢查無 `gt_private/`；當真實 W-C5 資料包尚未到位時，明確標記 `real_data_status: "UNKNOWN"` 並自動切換至內建 `SYNTHETIC` 測試資料集。
4. **Cell 4 — 中斷續跑（Checkpoint / Resume）實驗執行器**：逐事件、逐模型組合（M0/RULE_ONLY 與 M1～M4）寫入 JSONL checkpoint，已完成的項目於重跑時自動跳過。
5. **Cell 5 — 結果彙整與匯出**：產出彙整 CSV 表格、視覺化比較圖（PNG）與匯出包 `EXPORT_MANIFEST.json` SHA-256。


## Cell 1：環境自檢（OS、CPU、RAM、GPU/VRAM、Python/torch 版本）與 `amidst.experiment-env.v1` 驗證

檢查執行環境硬體與套件版本，並透過 `scripts/research/experiment_env.py` 驗證產出的 `amidst.experiment-env.v1` manifest 與計算正規化 `env_manifest_sha256`。


In [1]:
# Cell 1：環境自檢（OS、CPU、RAM、GPU/VRAM、Python/torch 版本）
# 產出並驗證符合 amidst.experiment-env.v1 契約之環境紀錄與 SHA-256
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts/research/colab_oneclick_runner.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "scripts/research") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts/research"))
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import colab_oneclick_runner as runner

WORKDIR = Path(os.environ.get("AMIDST_COLAB_WORKDIR", "/tmp/amidst_colab_oneclick"))
WORKDIR.mkdir(parents=True, exist_ok=True)

env_report = runner.inspect_environment(deterministic=True)
print(json.dumps({
    "schema_version": env_report["env_manifest"]["schema_version"],
    "env_manifest_sha256": env_report["env_manifest_sha256"],
    "validation_errors": env_report["validation_errors"],
    "probe_summary": env_report["probe_summary"],
}, ensure_ascii=False, indent=2))


{
  "schema_version": "amidst.experiment-env.v1",
  "env_manifest_sha256": "bca0694cfa283dbaded84aa17b2caef18c875a3a657f198f9c718faa38ec49bb",
  "validation_errors": [],
  "probe_summary": {
    "in_colab": false,
    "os_name": "SYNTHETIC-Colab-Linux",
    "cpu_cores": "4",
    "ram_gib": 12.7,
    "gpu_model": "NONE",
    "gpu_vram_gib": "NONE",
    "python_version": "3.12.0",
    "torch_version": "NOT_INSTALLED"
  }
}


## Cell 2：API Key 安全讀取（Colab Secrets / 環境變數狀態檢查）

僅從 `google.colab.userdata` 或環境變數讀取金鑰是否存在（回報 `PRESENT` 或 `MISSING`），**絕不印出、回傳或寫入任何金鑰明文或片段**。


In [2]:
# Cell 2：API Key 安全讀取（僅檢查 google.colab.userdata 或環境變數狀態 PRESENT / MISSING）
# 安全規則：絕不印出或記錄任何 API Key 實際數值
key_report = runner.check_api_keys()
print(json.dumps(key_report, ensure_ascii=False, indent=2))


{
  "key_Names_checked": [
    "OPENAI_API_KEY",
    "GEMINI_API_KEY",
    "ANTHROPIC_API_KEY",
    "OLLAMA_HOST"
  ],
  "keys_status": {
    "OPENAI_API_KEY": "MISSING",
    "GEMINI_API_KEY": "MISSING",
    "ANTHROPIC_API_KEY": "MISSING",
    "OLLAMA_HOST": "MISSING"
  },
  "any_key_present": false,
  "secret_values_logged": false,
  "policy": "STATUS_ONLY_NEVER_LOG_VALUES"
}


## Cell 3：Google Drive 掛載與 W-C5 safe 資料包 SHA-256 驗證

若於 Colab 環境中執行則掛載 Google Drive，並檢查 W-C5 safe 資料包之檔案 SHA-256 與確認不含 `gt_private/` 路徑。當真實 W-C5 safe 資料包（#416）尚未提供時，明確標記 `real_data_status: "UNKNOWN"` 並自動切換至內建 `SYNTHETIC` 測試資料集，確保 `Run all` 全程可跑通。


In [3]:
# Cell 3：Google Drive 掛載與 W-C5 safe 資料包 SHA-256 驗證（含 gt_private 拒絕檢查）
# 當真實 W-C5 資料包尚未到位時，標記 real_data_status="UNKNOWN" 並切換至內建 SYNTHETIC 資料集
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

WC5_PACK_DIR = os.environ.get("AMIDST_WC5_SAFE_PACK_DIR", "/content/drive/MyDrive/amidst_wc5_safe_pack")
data_pack_report = runner.verify_data_pack(pack_dir=WC5_PACK_DIR, allow_synthetic_fallback=True)
print(json.dumps({
    "real_data_status": data_pack_report["real_data_status"],
    "real_data_note": data_pack_report["real_data_note"],
    "active_data_mode": data_pack_report["active_data_mode"],
    "source_class": data_pack_report["source_class"],
    "gt_path_scan": data_pack_report["gt_path_scan"],
    "sha256_verified": data_pack_report["sha256_verified"],
    "pack_manifest_sha256": data_pack_report["pack_manifest_sha256"],
    "file_count": len(data_pack_report["files_verified"]),
}, ensure_ascii=False, indent=2))


{
  "real_data_status": "UNKNOWN",
  "real_data_note": "UNKNOWN (awaiting #416 W-C5 safe pack)",
  "active_data_mode": "SYNTHETIC",
  "source_class": "SYNTHETIC",
  "gt_path_scan": "PASS",
  "sha256_verified": true,
  "pack_manifest_sha256": "8e528d4a7dacb4745b77af38a4a5b867c57f850c59fdb5b8ec6507af237e38b6",
  "file_count": 6
}


## Cell 4：中斷續跑（Checkpoint / Resume）實驗執行器（M0/RULE_ONLY 與 M1～M4）

逐事件、逐模型組合將完成結果寫入 JSONL checkpoint。以下先執行前 10 個單元模擬中斷，再立即續跑剩餘單元，驗證已完成且設定／輸入 SHA-256 一致的項目會自動跳過不重算。


In [4]:
# Cell 4：中斷續跑（Checkpoint / Resume）實驗執行器
# 逐事件、逐模型組合（M0/RULE_ONLY 與 M1~M4）寫入 JSONL checkpoint；若已存在相同設定與輸入 hash 則自動跳過
checkpoint_file = WORKDIR / "checkpoints/experiment_checkpoint.jsonl"
if checkpoint_file.exists():
    checkpoint_file.unlink()

config_sha = env_report["env_manifest"]["parameter_set"]["parameter_set_sha256"]

# 第一階段：模擬執行 10 個單元後中斷
partial_run = runner.run_checkpointed_experiments(
    checkpoint_file,
    config_sha256=config_sha,
    max_units_to_run=10,
)
# 第二階段：從同一個 checkpoint 續跑完成全部剩餘單元（已完成的 10 個單元自動跳過）
checkpoint_report = runner.run_checkpointed_experiments(
    checkpoint_file,
    config_sha256=config_sha,
    max_units_to_run=None,
)
print(json.dumps({
    "partial_run": {
        "executed_now": partial_run["executed_now"],
        "skipped_from_checkpoint": partial_run["skipped_from_checkpoint"],
        "interrupted": partial_run["interrupted"],
    },
    "resumed_run": {
        "total_units": checkpoint_report["total_units"],
        "skipped_from_checkpoint": checkpoint_report["skipped_from_checkpoint"],
        "executed_now": checkpoint_report["executed_now"],
        "completed_total": checkpoint_report["completed_total"],
        "interrupted": checkpoint_report["interrupted"],
        "checkpoint_sha256": checkpoint_report["checkpoint_sha256"],
        "completed_by_experiment": checkpoint_report["completed_by_experiment"],
    },
}, ensure_ascii=False, indent=2))


{
  "partial_run": {
    "executed_now": 10,
    "skipped_from_checkpoint": 0,
    "interrupted": true
  },
  "resumed_run": {
    "total_units": 251,
    "skipped_from_checkpoint": 10,
    "executed_now": 241,
    "completed_total": 251,
    "interrupted": false,
    "checkpoint_sha256": "49a4d35ee7329ed83a80966049089d4c9798452a3acf10c9f5c1336347b23392",
    "completed_by_experiment": {
      "M0_BASELINE": 10,
      "M1_AGENT_VS_RULE": 56,
      "M2_SCORER_V2": 17,
      "M3_Q3_PARSER": 72,
      "M4_MODEL_COMPARE_Q2": 48,
      "M4_MODEL_COMPARE_Q3": 48
    }
  }
}


## Cell 5：結果彙整與匯出 CSV 表格、視覺化比較圖（PNG）與匯出包 MANIFEST SHA-256

呼叫研究分析模組產出 M0～M4 各模組 CSV 表格與 PNG/SVG 視覺化圖表，並彙整總表 `colab_oneclick_summary.csv`、總覽比較圖 `colab_oneclick_summary.png` 與 `EXPORT_MANIFEST.json`。


In [5]:
# Cell 5：結果彙整與匯出 CSV 表格、視覺化比較圖（PNG）與匯出包 MANIFEST SHA-256
# 產出 M0~M4 統計表、比較圖與包含所有產物雜湊的 EXPORT_MANIFEST.json
export_report = runner.export_analysis_bundle(
    WORKDIR / "exports",
    env_manifest=env_report["env_manifest"],
    env_manifest_sha256=env_report["env_manifest_sha256"],
    data_pack_report=data_pack_report,
    checkpoint_report=checkpoint_report,
    bootstrap_b=200,
)
print(json.dumps({
    "real_data_status": export_report["real_data_status"],
    "source_class": export_report["source_class"],
    "file_count": export_report["file_count"],
    "summary_csv_sha256": export_report["summary_csv_sha256"],
    "summary_png_sha256": export_report["summary_png_sha256"],
    "export_manifest_sha256": export_report["export_manifest_sha256"],
    "summary_rows": export_report["summary_rows"],
}, ensure_ascii=False, indent=2))


{
  "real_data_status": "UNKNOWN",
  "source_class": "SYNTHETIC",
  "file_count": 42,
  "summary_csv_sha256": "9a2f0f08f9bf09e761a1030dec98110a4f6ac71ee67c80a2dc273c1736514c17",
  "summary_png_sha256": "2b01ca30f8bcd43ee25d972f7a7d4b830a720b13ad438ed94c8d044a8c6861cf",
  "export_manifest_sha256": "3bdb3c4d9990a55c6bf3259637971ccf8d2ed78d4903e7df1a435291ac9768a3",
  "summary_rows": [
    {
      "module": "M0_BASELINE",
      "arm_or_scope": "RULE_ONLY (TOTAL:ALL)",
      "primary_metric": "C_evt_pct",
      "score_pct": "57.1429",
      "ci_lo": "50.0000",
      "ci_hi": "66.6667",
      "n_scored": "7",
      "source_class": "SYNTHETIC",
      "real_data_status": "UNKNOWN",
      "env_manifest_sha256": "bca0694cfa283dbaded84aa17b2caef18c875a3a657f198f9c718faa38ec49bb"
    },
    {
      "module": "M1_Q5_SWITCH",
      "arm_or_scope": "RULE_ONLY",
      "primary_metric": "C_evt_pct",
      "score_pct": "66.6667",
      "ci_lo": "66.6667",
      "ci_hi": "66.6667",
      "n_scored": "18